In [1]:
# Cell 1: Imports and project paths

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Get current working directory
CURRENT_DIR = Path.cwd()

# Detect project root
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
elif (CURRENT_DIR / "datasets").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "datasets").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


# Define paths
DATA_PATH = PROJECT_ROOT / "datasets"
PROCESSED_DATA_PATH = DATA_PATH / "processed"
REPORTS_PATH = PROJECT_ROOT / "reports"

RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DATA_PATH)

Project root: /Users/adityachaubey/Desktop/RetailAI
Processed data: /Users/adityachaubey/Desktop/RetailAI/datasets/processed


In [2]:
# Cell 2: Load customer segmentation results

customer_segments = pd.read_csv(
    PROCESSED_DATA_PATH / "customer_segments.csv"
)

print(
    "Customer segmentation shape:",
    customer_segments.shape
)

display(customer_segments.head())

Customer segmentation shape: (93358, 13)


,customer_unique_id,recency,frequency,monetary,aov,customer_lifetime_days,orders_per_30_days,R_score,F_score,M_score,RFM_score,cluster,customer_segment
0,6469f99c1f9dfae7733b25662e7f1782,62,7,758.83,108.404286,281,0.747331,5,5,5,15,0,Repeat High-Value Customers
1,8d50f5eadf50201ccdcedfb9e2ac8455,8,15,879.27,58.618000,427,1.053864,5,5,5,15,0,Repeat High-Value Customers
2,dc813062e0fc23409cd255f7f53c7074,6,6,1094.63,182.438333,417,0.431655,5,5,5,15,0,Repeat High-Value Customers
3,ca77025e7201e3b30c44b472ff346268,89,7,1122.72,160.388571,234,0.897436,5,5,5,15,0,Repeat High-Value Customers
4,b896655e2083a1d76b7b85df8fc86e40,54,4,507.06,126.765000,283,0.424028,5,4,5,14,0,Repeat High-Value Customers


In [3]:
# Cell 3: Load supporting datasets

orders = pd.read_parquet(
    PROCESSED_DATA_PATH / "orders.parquet"
)

order_items = pd.read_parquet(
    PROCESSED_DATA_PATH / "order_items.parquet"
)

order_reviews = pd.read_parquet(
    PROCESSED_DATA_PATH / "order_reviews.parquet"
)

products = pd.read_parquet(
    PROCESSED_DATA_PATH / "products.parquet"
)

category_translation = pd.read_parquet(
    PROCESSED_DATA_PATH / "category_translation.parquet"
)

customers = pd.read_parquet(
    PROCESSED_DATA_PATH / "customers.parquet"
)

print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Reviews:", order_reviews.shape)
print("Products:", products.shape)

Orders: (99441, 8)
Order Items: (112650, 7)
Reviews: (99224, 7)
Products: (32951, 9)


In [4]:
# Cell 4: Build customer satisfaction features

# Merge reviews with customer identifiers
customer_reviews = order_reviews[
    [
        "order_id",
        "review_score"
    ]
].merge(
    orders[
        [
            "order_id",
            "customer_id"
        ]
    ],
    on="order_id",
    how="left"
).merge(
    customers[
        [
            "customer_id",
            "customer_unique_id"
        ]
    ],
    on="customer_id",
    how="left"
)

# Aggregate review behavior
review_features = (
    customer_reviews
    .groupby("customer_unique_id")
    .agg(
        avg_review_score=(
            "review_score",
            "mean"
        ),

        review_count=(
            "review_score",
            "count"
        ),

        low_rating_percentage=(
            "review_score",
            lambda x: (x <= 2).mean() * 100
        )
    )
    .reset_index()
)

display(review_features.head())

,customer_unique_id,avg_review_score,review_count,low_rating_percentage
0,0000366f3b9a7992bf8c76cfdf3221e2,5.0,1,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4.0,1,0.0
2,0000f46a3911fa3c0805444483337064,3.0,1,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,4.0,1,0.0
4,0004aac84e0df4da2b147fca70cf8255,5.0,1,0.0


In [5]:
# Cell 5: Combine customer segmentation with satisfaction data

customer_profile = customer_segments.merge(
    review_features,
    on="customer_unique_id",
    how="left"
)

# Fill customers without reviews
customer_profile["avg_review_score"] = (
    customer_profile["avg_review_score"]
    .fillna(0)
)

customer_profile["low_rating_percentage"] = (
    customer_profile["low_rating_percentage"]
    .fillna(0)
)

display(customer_profile.head())

,customer_unique_id,recency,frequency,monetary,aov,customer_lifetime_days,orders_per_30_days,R_score,F_score,M_score,RFM_score,cluster,customer_segment,avg_review_score,review_count,low_rating_percentage
0,6469f99c1f9dfae7733b25662e7f1782,62,7,758.83,108.404286,281,0.747331,5,5,5,15,0,Repeat High-Value Customers,5.000000,7.0,0.000000
1,8d50f5eadf50201ccdcedfb9e2ac8455,8,15,879.27,58.618000,427,1.053864,5,5,5,15,0,Repeat High-Value Customers,4.764706,17.0,5.882353
2,dc813062e0fc23409cd255f7f53c7074,6,6,1094.63,182.438333,417,0.431655,5,5,5,15,0,Repeat High-Value Customers,4.500000,6.0,0.000000
3,ca77025e7201e3b30c44b472ff346268,89,7,1122.72,160.388571,234,0.897436,5,5,5,15,0,Repeat High-Value Customers,5.000000,7.0,0.000000
4,b896655e2083a1d76b7b85df8fc86e40,54,4,507.06,126.765000,283,0.424028,5,4,5,14,0,Repeat High-Value Customers,5.000000,4.0,0.000000


In [6]:
# Cell 6: Identify the current customer state

def identify_customer_state(row):

    # Strong repeat customer
    if (
        row["frequency"] >= 2
        and row["RFM_score"] >= 9
        and row["recency"] <= 180
    ):
        return "Active Repeat Customer"

    # Repeat customer who has become inactive
    elif (
        row["frequency"] >= 2
        and row["recency"] > 180
    ):
        return "Repeat Customer At Risk"

    # High-value one-time customer
    elif (
        row["frequency"] == 1
        and row["monetary"] >= customer_profile["monetary"].quantile(0.75)
        and row["recency"] <= 180
    ):
        return "High-Value One-Time Customer"

    # Recently acquired one-time customer
    elif (
        row["frequency"] == 1
        and row["recency"] <= 90
    ):
        return "New One-Time Customer"

    # Old one-time customer
    elif (
        row["frequency"] == 1
        and row["recency"] > 180
    ):
        return "Dormant One-Time Customer"

    return "Developing Customer"


customer_profile["customer_state"] = (
    customer_profile.apply(
        identify_customer_state,
        axis=1
    )
)

display(
    customer_profile[
        [
            "customer_unique_id",
            "customer_segment",
            "customer_state",
            "recency",
            "frequency",
            "monetary",
            "RFM_score"
        ]
    ].head(20)
)

,customer_unique_id,customer_segment,customer_state,recency,frequency,monetary,RFM_score
0,6469f99c1f9dfae7733b25662e7f1782,Repeat High-Value Customers,Active Repeat Customer,62,7,758.83,15
1,8d50f5eadf50201ccdcedfb9e2ac8455,Repeat High-Value Customers,Active Repeat Customer,8,15,879.27,15
2,dc813062e0fc23409cd255f7f53c7074,Repeat High-Value Customers,Active Repeat Customer,6,6,1094.63,15
3,ca77025e7201e3b30c44b472ff346268,Repeat High-Value Customers,Active Repeat Customer,89,7,1122.72,15
4,b896655e2083a1d76b7b85df8fc86e40,Repeat High-Value Customers,Active Repeat Customer,54,4,507.06,14
5,4facc2e6fbc2bffab2fea92d2b4aa7e4,Repeat High-Value Customers,Active Repeat Customer,15,4,1760.75,14
6,795c1622cf7a53d63d324e862349d01c,Repeat High-Value Customers,Active Repeat Customer,11,4,354.23,14
7,394ac4de8f3acb14253c177f0e15bc58,Repeat High-Value Customers,Active Repeat Customer,13,5,745.41,14
8,fe81bb32c243a86b2f86fbf053fe6140,Repeat High-Value Customers,Active Repeat Customer,69,5,1590.76,14
9,397b44d5bb99eabf54ea9c2b41ebb905,Repeat High-Value Customers,Active Repeat Customer,72,4,1756.53,14


In [7]:
# Cell 7: Identify customers requiring service recovery

customer_profile["service_recovery_flag"] = (
    (customer_profile["avg_review_score"] > 0)
    & (customer_profile["avg_review_score"] <= 2)
)

print(
    "Customers requiring service recovery:",
    customer_profile["service_recovery_flag"].sum()
)

display(
    customer_profile[
        customer_profile["service_recovery_flag"]
    ][
        [
            "customer_unique_id",
            "customer_state",
            "avg_review_score",
            "low_rating_percentage",
            "monetary"
        ]
    ].head(10)
)

Customers requiring service recovery: 11765


,customer_unique_id,customer_state,avg_review_score,low_rating_percentage,monetary
39,33176de67c05eeed870fd49f234387a0,Active Repeat Customer,2.000000,75.0,1254.01
76,d26c616e241736e0c1c1ab14150239e7,Active Repeat Customer,1.666667,100.0,579.78
107,1f98d2384ff7a372e6a2d3bb75cbcd54,Active Repeat Customer,1.000000,100.0,1381.41
119,bba29bdc2bd4deeb7bba3c61ea98e012,Active Repeat Customer,1.000000,100.0,367.12
126,8f567321ba12101dedc07c70b7021069,Active Repeat Customer,2.000000,50.0,296.40
142,7588c20c457369de8caad958afc1cc06,Active Repeat Customer,1.333333,100.0,265.84
202,78a038d19f31c40c5dd68c112ca00ad1,Active Repeat Customer,1.000000,100.0,242.41
217,83476105a4e9a12fcddd09081dc6cefe,Active Repeat Customer,1.500000,100.0,283.56
218,53e4d5ab8287848319f0220d47ee7ca0,Active Repeat Customer,2.000000,50.0,232.23
254,65571628698194dbda526c22f5ed2317,Active Repeat Customer,1.000000,100.0,418.94


In [8]:
# Cell 8: Define the retention strategy library

strategy_library = {
    
    "SERVICE_RECOVERY": {
        "recommended_action": "Resolve dissatisfaction and rebuild customer trust",
        "channel": "Customer Support + Email",
        "incentive": "Goodwill benefit if appropriate",
        "business_goal": "Recover dissatisfied customers"
    },

    "SECOND_PURCHASE": {
        "recommended_action": "Recommend relevant products for the customer's second purchase",
        "channel": "Email + App",
        "incentive": "Optional first-repeat incentive",
        "business_goal": "Convert one-time customers into repeat customers"
    },

    "VIP_RECOGNITION": {
        "recommended_action": "Provide early access, exclusive benefits and personalized recommendations",
        "channel": "Email + App",
        "incentive": "Exclusive loyalty benefit",
        "business_goal": "Increase loyalty and customer lifetime value"
    },

    "CROSS_SELL": {
        "recommended_action": "Recommend complementary products based on previous purchases",
        "channel": "Email + Homepage + App",
        "incentive": "No discount required initially",
        "business_goal": "Increase purchase frequency"
    },

    "BASKET_EXPANSION": {
        "recommended_action": "Recommend related products or product bundles",
        "channel": "Checkout + Email",
        "incentive": "Bundle benefit if required",
        "business_goal": "Increase average order value"
    },

    "WIN_BACK": {
        "recommended_action": "Re-engage the customer with personalized products and new arrivals",
        "channel": "Email + Retargeting",
        "incentive": "Targeted incentive only if required",
        "business_goal": "Reactivate inactive customers"
    },

    "LOYALTY_HABIT": {
        "recommended_action": "Create repeat-purchase habits using personalized discovery and reminders",
        "channel": "Push + Email",
        "incentive": "Loyalty points",
        "business_goal": "Increase purchase frequency"
    },

    "ADVOCACY": {
        "recommended_action": "Encourage reviews, referrals and customer advocacy",
        "channel": "Email + App",
        "incentive": "Referral or loyalty benefit",
        "business_goal": "Increase retention and referrals"
    }
}

print("Number of strategies:", len(strategy_library))

Number of strategies: 8


In [9]:
# Cell 9: Select the next-best retention strategy for each customer

def select_next_best_action(row):

    # First priority: solve dissatisfaction before selling anything
    if row["service_recovery_flag"]:
        return "SERVICE_RECOVERY"

    # High-value and highly engaged customers
    elif (
        row["customer_state"] == "Active Repeat Customer"
        and row["RFM_score"] >= 10
    ):
        return "VIP_RECOGNITION"

    # Previously valuable repeat customers who became inactive
    elif row["customer_state"] == "Repeat Customer At Risk":
        return "WIN_BACK"

    # Valuable first-time customers
    elif row["customer_state"] == "High-Value One-Time Customer":
        return "SECOND_PURCHASE"

    # Recent first-time customers
    elif row["customer_state"] == "New One-Time Customer":
        return "SECOND_PURCHASE"

    # Inactive one-time customers
    elif row["customer_state"] == "Dormant One-Time Customer":
        return "WIN_BACK"

    # Existing repeat customers
    elif row["frequency"] >= 2:
        return "CROSS_SELL"

    # Default action
    else:
        return "LOYALTY_HABIT"


customer_profile["strategy"] = customer_profile.apply(
    select_next_best_action,
    axis=1
)

print("Strategy distribution:")
display(
    customer_profile["strategy"]
    .value_counts()
    .rename_axis("strategy")
    .reset_index(name="customers")
)

Strategy distribution:


,strategy,customers
0,WIN_BACK,47871
1,SECOND_PURCHASE,20063
2,LOYALTY_HABIT,12487
3,SERVICE_RECOVERY,11765
4,VIP_RECOGNITION,1021
5,CROSS_SELL,151


In [10]:
# Cell 10: Attach business strategy details to each customer

customer_profile["recommended_action"] = (
    customer_profile["strategy"]
    .map(lambda x: strategy_library[x]["recommended_action"])
)

customer_profile["channel"] = (
    customer_profile["strategy"]
    .map(lambda x: strategy_library[x]["channel"])
)

customer_profile["incentive"] = (
    customer_profile["strategy"]
    .map(lambda x: strategy_library[x]["incentive"])
)

customer_profile["business_goal"] = (
    customer_profile["strategy"]
    .map(lambda x: strategy_library[x]["business_goal"])
)

display(
    customer_profile[
        [
            "customer_unique_id",
            "customer_state",
            "strategy",
            "recommended_action",
            "channel",
            "incentive",
            "business_goal"
        ]
    ].head(10)
)

,customer_unique_id,customer_state,strategy,recommended_action,channel,incentive,business_goal
0,6469f99c1f9dfae7733b25662e7f1782,Active Repeat Customer,VIP_RECOGNITION,"Provide early access, exclusive benefits and p...",Email + App,Exclusive loyalty benefit,Increase loyalty and customer lifetime value
1,8d50f5eadf50201ccdcedfb9e2ac8455,Active Repeat Customer,VIP_RECOGNITION,"Provide early access, exclusive benefits and p...",Email + App,Exclusive loyalty benefit,Increase loyalty and customer lifetime value
2,dc813062e0fc23409cd255f7f53c7074,Active Repeat Customer,VIP_RECOGNITION,"Provide early access, exclusive benefits and p...",Email + App,Exclusive loyalty benefit,Increase loyalty and customer lifetime value
3,ca77025e7201e3b30c44b472ff346268,Active Repeat Customer,VIP_RECOGNITION,"Provide early access, exclusive benefits and p...",Email + App,Exclusive loyalty benefit,Increase loyalty and customer lifetime value
4,b896655e2083a1d76b7b85df8fc86e40,Active Repeat Customer,VIP_RECOGNITION,"Provide early access, exclusive benefits and p...",Email + App,Exclusive loyalty benefit,Increase loyalty and customer lifetime value
5,4facc2e6fbc2bffab2fea92d2b4aa7e4,Active Repeat Customer,VIP_RECOGNITION,"Provide early access, exclusive benefits and p...",Email + App,Exclusive loyalty benefit,Increase loyalty and customer lifetime value
6,795c1622cf7a53d63d324e862349d01c,Active Repeat Customer,VIP_RECOGNITION,"Provide early access, exclusive benefits and p...",Email + App,Exclusive loyalty benefit,Increase loyalty and customer lifetime value
7,394ac4de8f3acb14253c177f0e15bc58,Active Repeat Customer,VIP_RECOGNITION,"Provide early access, exclusive benefits and p...",Email + App,Exclusive loyalty benefit,Increase loyalty and customer lifetime value
8,fe81bb32c243a86b2f86fbf053fe6140,Active Repeat Customer,VIP_RECOGNITION,"Provide early access, exclusive benefits and p...",Email + App,Exclusive loyalty benefit,Increase loyalty and customer lifetime value
9,397b44d5bb99eabf54ea9c2b41ebb905,Active Repeat Customer,VIP_RECOGNITION,"Provide early access, exclusive benefits and p...",Email + App,Exclusive loyalty benefit,Increase loyalty and customer lifetime value


In [11]:
# Cell 11: Calculate business benchmarks

median_aov = customer_profile["aov"].median()
median_monetary = customer_profile["monetary"].median()
median_frequency = customer_profile["frequency"].median()
median_recency = customer_profile["recency"].median()

high_value_threshold = (
    customer_profile["monetary"].quantile(0.75)
)

repeat_customers = customer_profile[
    customer_profile["frequency"] >= 2
]

repeat_customer_rate = (
    len(repeat_customers) /
    len(customer_profile)
)

print(f"Median AOV: ₹{median_aov:,.2f}")
print(f"Median Customer Value: ₹{median_monetary:,.2f}")
print(f"Median Purchase Frequency: {median_frequency:.2f}")
print(f"Median Recency: {median_recency:.2f} days")
print(f"High-Value Threshold: ₹{high_value_threshold:,.2f}")
print(f"Repeat Customer Rate: {repeat_customer_rate:.2%}")

Median AOV: ₹105.63
Median Customer Value: ₹107.78
Median Purchase Frequency: 1.00
Median Recency: 218.00 days
High-Value Threshold: ₹182.56
Repeat Customer Rate: 3.00%


In [12]:
# Cell 12: Calculate customer value and retention opportunity

# Historical customer value
customer_profile["historical_customer_value"] = (
    customer_profile["monetary"]
)

# Historical average order value
customer_profile["historical_aov"] = (
    customer_profile["aov"]
)

# Potential value of one additional order
customer_profile["potential_next_order_value"] = (
    customer_profile["aov"]
)

# Compare customer's value with the typical customer
customer_profile["value_multiplier"] = (
    customer_profile["monetary"] /
    median_monetary
)

# Compare customer's AOV with the typical AOV
customer_profile["aov_multiplier"] = (
    customer_profile["aov"] /
    median_aov
)

display(
    customer_profile[
        [
            "customer_unique_id",
            "frequency",
            "monetary",
            "aov",
            "value_multiplier",
            "aov_multiplier",
            "potential_next_order_value"
        ]
    ].head(10)
)

,customer_unique_id,frequency,monetary,aov,value_multiplier,aov_multiplier,potential_next_order_value
0,6469f99c1f9dfae7733b25662e7f1782,7,758.83,108.404286,7.040546,1.026264,108.404286
1,8d50f5eadf50201ccdcedfb9e2ac8455,15,879.27,58.618000,8.158007,0.554937,58.618000
2,dc813062e0fc23409cd255f7f53c7074,6,1094.63,182.438333,10.156151,1.727145,182.438333
3,ca77025e7201e3b30c44b472ff346268,7,1122.72,160.388571,10.416775,1.518400,160.388571
4,b896655e2083a1d76b7b85df8fc86e40,4,507.06,126.765000,4.704583,1.200085,126.765000
5,4facc2e6fbc2bffab2fea92d2b4aa7e4,4,1760.75,440.187500,16.336519,4.167258,440.187500
6,795c1622cf7a53d63d324e862349d01c,4,354.23,88.557500,3.286602,0.838375,88.557500
7,394ac4de8f3acb14253c177f0e15bc58,5,745.41,149.082000,6.916033,1.411360,149.082000
8,fe81bb32c243a86b2f86fbf053fe6140,5,1590.76,318.152000,14.759325,3.011947,318.152000
9,397b44d5bb99eabf54ea9c2b41ebb905,4,1756.53,439.132500,16.297365,4.157271,439.132500


In [13]:
# Cell 13: Calculate customer targeting priority

customer_profile["value_percentile"] = (
    customer_profile["monetary"]
    .rank(pct=True) * 100
)

customer_profile["rfm_percentile"] = (
    customer_profile["RFM_score"]
    .rank(pct=True) * 100
)

customer_profile["priority_score"] = (
    customer_profile["value_percentile"] * 0.40
    + customer_profile["rfm_percentile"] * 0.30
    + np.where(
        customer_profile["customer_state"] == "Repeat Customer At Risk",
        20,
        0
    )
    + np.where(
        customer_profile["customer_state"] == "High-Value One-Time Customer",
        15,
        0
    )
    + np.where(
        customer_profile["service_recovery_flag"],
        25,
        0
    )
)

customer_profile["priority_score"] = (
    customer_profile["priority_score"]
    .clip(upper=100)
    .round(2)
)

display(
    customer_profile[
        [
            "customer_unique_id",
            "customer_state",
            "monetary",
            "RFM_score",
            "strategy",
            "priority_score"
        ]
    ]
    .sort_values(
        "priority_score",
        ascending=False
    )
    .head(15)
)

,customer_unique_id,customer_state,monetary,RFM_score,strategy,priority_score
4748,a5a41ed175e9ced7ee47aa1d2680b523,High-Value One-Time Customer,614.77,10,SERVICE_RECOVERY,100.0
8794,037f1240d3a139aef654264b518eb7e1,High-Value One-Time Customer,338.85,10,SERVICE_RECOVERY,100.0
833,d2f0b9e0fbf779c60ff56f493ca47cf1,High-Value One-Time Customer,639.43,11,SERVICE_RECOVERY,100.0
8791,d2801e87e161b485eb85e914c2524d3f,High-Value One-Time Customer,257.19,10,SERVICE_RECOVERY,100.0
21411,12a58098f27b00d10cf51bc6f9e4c261,High-Value One-Time Customer,401.30,9,SERVICE_RECOVERY,100.0
8763,cd2671a21d95e72a623e07b63551335c,High-Value One-Time Customer,260.24,10,SERVICE_RECOVERY,100.0
2458,61e55a4c5ac0890968648505e89a0dd5,High-Value One-Time Customer,232.81,11,SERVICE_RECOVERY,100.0
8760,f4c6d8edc2ed35807ec325b287b190e3,High-Value One-Time Customer,364.65,10,SERVICE_RECOVERY,100.0
11590,32752da00d66a31189201fc0772f600b,High-Value One-Time Customer,342.15,10,SERVICE_RECOVERY,100.0
2465,2d20a91bda6cce68319de3919a613914,High-Value One-Time Customer,665.44,11,SERVICE_RECOVERY,100.0


In [14]:
# Cell 14: Generate an automatic business explanation

def generate_business_reason(row):

    strategy = row["strategy"]
    revenue = row["monetary"]
    aov = row["aov"]
    frequency = row["frequency"]
    recency = row["recency"]
    rfm = row["RFM_score"]
    review = row["avg_review_score"]

    if strategy == "SERVICE_RECOVERY":

        return (
            f"The customer has an average review score of {review:.1f}/5, "
            f"indicating potential dissatisfaction. The customer has "
            f"already generated ₹{revenue:,.2f} in historical revenue. "
            f"Therefore, the business should prioritize service recovery "
            f"before attempting another sale."
        )

    elif strategy == "VIP_RECOGNITION":

        return (
            f"The customer has made {frequency:.0f} purchases and generated "
            f"₹{revenue:,.2f} in historical revenue. Their RFM score is "
            f"{rfm:.0f}, indicating strong engagement and customer value. "
            f"VIP treatment is recommended to protect this high-value "
            f"relationship without depending on heavy discounts."
        )

    elif strategy == "WIN_BACK":

        return (
            f"The customer previously purchased {frequency:.0f} times but "
            f"has been inactive for {recency:.0f} days. This indicates "
            f"potential customer attrition. Since the customer has already "
            f"demonstrated willingness to purchase, reactivation can be "
            f"more targeted than acquiring a completely new customer."
        )

    elif strategy == "SECOND_PURCHASE":

        return (
            f"The customer has purchased only once and generated "
            f"₹{revenue:,.2f}. The primary objective is to convert this "
            f"one-time transaction into a repeat relationship. A second "
            f"purchase can increase customer lifetime value."
        )

    elif strategy == "CROSS_SELL":

        return (
            f"The customer has already made {frequency:.0f} purchases, "
            f"showing repeat behaviour. Instead of immediately using a "
            f"discount, complementary products can be recommended to "
            f"increase future purchase frequency and basket value."
        )

    elif strategy == "LOYALTY_HABIT":

        return (
            f"The customer currently shows limited repeat behaviour. "
            f"The recommended approach is to build a purchasing habit "
            f"through personalized discovery, reminders and loyalty "
            f"benefits rather than immediately relying on discounts."
        )

    else:

        return (
            f"The customer has demonstrated purchasing activity. "
            f"The recommended strategy is designed to increase future "
            f"customer value through targeted engagement."
        )


customer_profile["business_reason"] = customer_profile.apply(
    generate_business_reason,
    axis=1
)

display(
    customer_profile[
        [
            "customer_unique_id",
            "strategy",
            "business_reason"
        ]
    ].head(10)
)

,customer_unique_id,strategy,business_reason
0,6469f99c1f9dfae7733b25662e7f1782,VIP_RECOGNITION,The customer has made 7 purchases and generate...
1,8d50f5eadf50201ccdcedfb9e2ac8455,VIP_RECOGNITION,The customer has made 15 purchases and generat...
2,dc813062e0fc23409cd255f7f53c7074,VIP_RECOGNITION,The customer has made 6 purchases and generate...
3,ca77025e7201e3b30c44b472ff346268,VIP_RECOGNITION,The customer has made 7 purchases and generate...
4,b896655e2083a1d76b7b85df8fc86e40,VIP_RECOGNITION,The customer has made 4 purchases and generate...
5,4facc2e6fbc2bffab2fea92d2b4aa7e4,VIP_RECOGNITION,The customer has made 4 purchases and generate...
6,795c1622cf7a53d63d324e862349d01c,VIP_RECOGNITION,The customer has made 4 purchases and generate...
7,394ac4de8f3acb14253c177f0e15bc58,VIP_RECOGNITION,The customer has made 5 purchases and generate...
8,fe81bb32c243a86b2f86fbf053fe6140,VIP_RECOGNITION,The customer has made 5 purchases and generate...
9,397b44d5bb99eabf54ea9c2b41ebb905,VIP_RECOGNITION,The customer has made 4 purchases and generate...


In [15]:
# Cell 15: Generate a mathematical explanation for the strategy

def generate_math_explanation(row):

    revenue = row["monetary"]
    frequency = row["frequency"]
    aov = row["aov"]
    recency = row["recency"]
    value_multiplier = row["value_multiplier"]

    if row["strategy"] == "SERVICE_RECOVERY":

        return (
            f"Historical customer value = ₹{revenue:,.2f}. "
            f"Historical AOV = ₹{aov:,.2f}. "
            f"The customer already represents an existing revenue "
            f"relationship, so protecting that relationship is the "
            f"primary financial objective."
        )

    elif row["strategy"] == "VIP_RECOGNITION":

        return (
            f"Customer revenue = ₹{revenue:,.2f} across "
            f"{frequency:.0f} orders. "
            f"AOV = ₹{aov:,.2f}. "
            f"The customer has generated {value_multiplier:.1f}× "
            f"the median customer value. "
            f"One additional order at historical AOV represents "
            f"approximately ₹{aov:,.2f} of potential transaction value."
        )

    elif row["strategy"] == "WIN_BACK":

        return (
            f"Historical customer value = ₹{revenue:,.2f}. "
            f"Historical AOV = ₹{aov:,.2f}. "
            f"The customer has been inactive for {recency:.0f} days. "
            f"One successful return purchase at historical AOV represents "
            f"approximately ₹{aov:,.2f} of potential transaction value."
        )

    elif row["strategy"] == "SECOND_PURCHASE":

        return (
            f"Current historical value = ₹{revenue:,.2f}. "
            f"Historical AOV = ₹{aov:,.2f}. "
            f"Converting the customer from 1 purchase to 2 purchases "
            f"would create approximately ₹{aov:,.2f} of additional "
            f"transaction value at the historical AOV."
        )

    elif row["strategy"] == "CROSS_SELL":

        return (
            f"Customer has already completed {frequency:.0f} orders. "
            f"Historical AOV = ₹{aov:,.2f}. "
            f"An additional purchase at historical AOV represents "
            f"approximately ₹{aov:,.2f} of potential transaction value."
        )

    elif row["strategy"] == "LOYALTY_HABIT":

        return (
            f"Historical AOV = ₹{aov:,.2f}. "
            f"An additional successful purchase represents approximately "
            f"₹{aov:,.2f} of potential transaction value. "
            f"The strategy therefore focuses on increasing repeat frequency."
        )

    else:

        return (
            f"Historical AOV = ₹{aov:,.2f}. "
            f"Future engagement can potentially increase customer value "
            f"above the current historical baseline."
        )


customer_profile["mathematical_explanation"] = customer_profile.apply(
    generate_math_explanation,
    axis=1
)

display(
    customer_profile[
        [
            "customer_unique_id",
            "strategy",
            "mathematical_explanation"
        ]
    ].head(10)
)

,customer_unique_id,strategy,mathematical_explanation
0,6469f99c1f9dfae7733b25662e7f1782,VIP_RECOGNITION,Customer revenue = ₹758.83 across 7 orders. AO...
1,8d50f5eadf50201ccdcedfb9e2ac8455,VIP_RECOGNITION,Customer revenue = ₹879.27 across 15 orders. A...
2,dc813062e0fc23409cd255f7f53c7074,VIP_RECOGNITION,"Customer revenue = ₹1,094.63 across 6 orders. ..."
3,ca77025e7201e3b30c44b472ff346268,VIP_RECOGNITION,"Customer revenue = ₹1,122.72 across 7 orders. ..."
4,b896655e2083a1d76b7b85df8fc86e40,VIP_RECOGNITION,Customer revenue = ₹507.06 across 4 orders. AO...
5,4facc2e6fbc2bffab2fea92d2b4aa7e4,VIP_RECOGNITION,"Customer revenue = ₹1,760.75 across 4 orders. ..."
6,795c1622cf7a53d63d324e862349d01c,VIP_RECOGNITION,Customer revenue = ₹354.23 across 4 orders. AO...
7,394ac4de8f3acb14253c177f0e15bc58,VIP_RECOGNITION,Customer revenue = ₹745.41 across 5 orders. AO...
8,fe81bb32c243a86b2f86fbf053fe6140,VIP_RECOGNITION,"Customer revenue = ₹1,590.76 across 5 orders. ..."
9,397b44d5bb99eabf54ea9c2b41ebb905,VIP_RECOGNITION,"Customer revenue = ₹1,756.53 across 4 orders. ..."


In [16]:
# Cell 16: Generate a simple executive recommendation

def generate_executive_recommendation(row):

    return (
        f"{row['strategy']} | "
        f"Priority: {row['priority_score']:.1f}/100 | "
        f"Potential next-order value: ₹{row['aov']:,.2f} | "
        f"Reason: {row['business_reason']} | "
        f"Math: {row['mathematical_explanation']}"
    )


customer_profile["executive_recommendation"] = (
    customer_profile.apply(
        generate_executive_recommendation,
        axis=1
    )
)

display(
    customer_profile[
        [
            "customer_unique_id",
            "strategy",
            "priority_score",
            "executive_recommendation"
        ]
    ].head(10)
)

,customer_unique_id,strategy,priority_score,executive_recommendation
0,6469f99c1f9dfae7733b25662e7f1782,VIP_RECOGNITION,69.14,VIP_RECOGNITION | Priority: 69.1/100 | Potenti...
1,8d50f5eadf50201ccdcedfb9e2ac8455,VIP_RECOGNITION,69.37,VIP_RECOGNITION | Priority: 69.4/100 | Potenti...
2,dc813062e0fc23409cd255f7f53c7074,VIP_RECOGNITION,69.60,VIP_RECOGNITION | Priority: 69.6/100 | Potenti...
3,ca77025e7201e3b30c44b472ff346268,VIP_RECOGNITION,69.62,VIP_RECOGNITION | Priority: 69.6/100 | Potenti...
4,b896655e2083a1d76b7b85df8fc86e40,VIP_RECOGNITION,68.21,VIP_RECOGNITION | Priority: 68.2/100 | Potenti...
5,4facc2e6fbc2bffab2fea92d2b4aa7e4,VIP_RECOGNITION,69.87,VIP_RECOGNITION | Priority: 69.9/100 | Potenti...
6,795c1622cf7a53d63d324e862349d01c,VIP_RECOGNITION,66.70,VIP_RECOGNITION | Priority: 66.7/100 | Potenti...
7,394ac4de8f3acb14253c177f0e15bc58,VIP_RECOGNITION,69.11,VIP_RECOGNITION | Priority: 69.1/100 | Potenti...
8,fe81bb32c243a86b2f86fbf053fe6140,VIP_RECOGNITION,69.83,VIP_RECOGNITION | Priority: 69.8/100 | Potenti...
9,397b44d5bb99eabf54ea9c2b41ebb905,VIP_RECOGNITION,69.87,VIP_RECOGNITION | Priority: 69.9/100 | Potenti...


In [17]:
# Cell 17: Create an overall business strategy summary

strategy_summary = (
    customer_profile
    .groupby("strategy")
    .agg(
        customers=("customer_unique_id", "count"),
        average_priority=("priority_score", "mean"),
        total_historical_value=("monetary", "sum"),
        average_aov=("aov", "mean"),
        potential_next_order_value=(
            "potential_next_order_value",
            "sum"
        )
    )
    .reset_index()
)

strategy_summary["customer_percentage"] = (
    strategy_summary["customers"]
    / len(customer_profile)
    * 100
)

strategy_summary = strategy_summary.sort_values(
    "potential_next_order_value",
    ascending=False
)

display(strategy_summary)

,strategy,customers,average_priority,total_historical_value,average_aov,potential_next_order_value,customer_percentage
5,WIN_BACK,47871,30.594580,7581824.88,153.724071,7.358925e+06,51.276805
2,SECOND_PURCHASE,20063,52.037470,4100447.79,204.378597,4.100448e+06,21.490392
3,SERVICE_RECOVERY,11765,63.908758,2251226.20,188.672448,2.219731e+06,12.602027
1,LOYALTY_HABIT,12487,31.529200,1114523.86,89.254734,1.114524e+06,13.375394
4,VIP_RECOGNITION,1021,62.268599,360464.62,162.650411,1.660661e+05,1.093640
0,CROSS_SELL,151,39.070397,13974.42,46.272914,6.987210e+03,0.161743


In [18]:
# Cell 18: Save the final retention strategy output

output_columns = [
    "customer_unique_id",
    "customer_segment",
    "customer_state",
    "recency",
    "frequency",
    "monetary",
    "aov",
    "RFM_score",
    "avg_review_score",
    "low_rating_percentage",
    "strategy",
    "recommended_action",
    "channel",
    "incentive",
    "business_goal",
    "priority_score",
    "potential_next_order_value",
    "value_percentile",
    "rfm_percentile",
    "business_reason",
    "mathematical_explanation",
    "executive_recommendation"
]

final_retention_output = customer_profile[
    output_columns
].copy()

REPORTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

final_retention_output.to_csv(
    PROCESSED_DATA_PATH / "customer_retention_strategies.csv",
    index=False
)

strategy_summary.to_csv(
    REPORTS_PATH / "retention_strategy_summary.csv",
    index=False
)

print("Customer retention strategies saved successfully.")
print("Strategy summary saved successfully.")

Customer retention strategies saved successfully.
Strategy summary saved successfully.
